# Create Training and Test Data for Fine-tuning

This notebook creates JSONL files for training and testing fine-tuned models.

## Overview
- Loads sections from `act_sections.csv` files
- Loads annotations from `unified_annotations.csv`
- Splits into 80/20 train/test ratio
- Ensures experimental sections (first 10) go to test set
- Creates JSONL files in OpenAI fine-tuning format


In [1]:
import json
import csv
import random
from pathlib import Path
from collections import defaultdict

# Set random seed for reproducibility
random.seed(42)

# Paths
base_dir = Path('.').resolve().parent
sections_csv_dir = base_dir / "Obligation_prohibition_picking_context"
unified_csv = base_dir / "automator" / "data" / "unified_annotations.csv"
prompt_path = base_dir / "automator" / "prompt.txt"
output_dir = base_dir / "automator" / "training_data"
output_dir.mkdir(parents=True, exist_ok=True)


## Step 1: Load Sections from CSV Files


In [2]:
# First, load annotations to know which sections we need
# Then load section JSON from hierarchy JSON files

# Load annotations first (this is our source of truth)
all_annotations = {}
with open(unified_csv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        legislation_id = row['legislation_id']
        annotations_json = json.loads(row['annotations'])
        all_annotations[legislation_id] = annotations_json

print(f"Loaded annotations for {len(all_annotations)} sections from unified CSV")

# Now load section JSON from hierarchy JSON files for sections that have annotations
all_legislation_ids = set(all_annotations.keys())

def extract_section_from_hierarchy(hierarchy_data, section_id):
    """Extract a single section from hierarchy JSON (recursive search)."""
    def find_section_recursive(data):
        if not isinstance(data, dict):
            return None
        
        # Check if section_id is a top-level key
        if section_id in data:
            return data[section_id]
        
        # Check parts -> sections structure
        if 'parts' in data:
            for part_key, part_data in data['parts'].items():
                if isinstance(part_data, dict):
                    # Check if sections are directly in part
                    if 'sections' in part_data and section_id in part_data['sections']:
                        return part_data['sections'][section_id]
                    # Check if chapters are in part
                    if 'chapters' in part_data:
                        for chapter_key, chapter_data in part_data['chapters'].items():
                            if isinstance(chapter_data, dict) and 'sections' in chapter_data:
                                if section_id in chapter_data['sections']:
                                    return chapter_data['sections'][section_id]
                    # Recursively search in part
                    result = find_section_recursive(part_data)
                    if result:
                        return result
        
        # Check direct sections
        if 'sections' in data and section_id in data['sections']:
            return data['sections'][section_id]
        
        # Check chapters -> sections
        if 'chapters' in data:
            for chapter_key, chapter_data in data['chapters'].items():
                if isinstance(chapter_data, dict) and 'sections' in chapter_data:
                    if section_id in chapter_data['sections']:
                        return chapter_data['sections'][section_id]
        
        return None
    
    return find_section_recursive(hierarchy_data)

# Group by act_id
from collections import defaultdict
act_sections = defaultdict(list)
for leg_id in all_legislation_ids:
    act_id = '_'.join(leg_id.split('_')[:2])
    section_id = '_'.join(leg_id.split('_')[2:])
    act_sections[act_id].append((leg_id, section_id))

# Load from hierarchy JSON files
all_sections = {}
not_found_sections = []

for act_id, section_list in act_sections.items():
    print(f"\nLoading sections for act {act_id} from hierarchy JSON files...")
    
    # Find chunk JSON files for this act
    chunk_files = sorted(sections_csv_dir.rglob(f"*{act_id}*chunk*.json"))
    
    if not chunk_files:
        print(f"  Warning: No chunk files found for {act_id}")
        continue
    
    print(f"  Found {len(chunk_files)} chunk files")
    
    # For each section, search through all chunk files
    for leg_id, section_id in section_list:
        if leg_id in all_sections:
            continue  # Already found
        
        found = False
        # Try each chunk file until we find the section
        for chunk_file in chunk_files:
            try:
                with open(chunk_file, 'r', encoding='utf-8') as f:
                    hierarchy_data = json.load(f)
                
                section_json = extract_section_from_hierarchy(hierarchy_data, section_id)
                if section_json:
                    all_sections[leg_id] = section_json
                    found = True
                    break  # Found it, move to next section
            except Exception as e:
                continue
        
        if not found:
            not_found_sections.append(leg_id)
            print(f"  Warning: Could not find {section_id} for {leg_id}")

print(f"\nLoaded {len(all_sections)} sections from hierarchy JSON files")

# Check which sections are in annotations but not in hierarchy JSON
annotations_only = set(all_annotations.keys()) - set(all_sections.keys())
if annotations_only:
    print(f"\n⚠️  {len(annotations_only)} sections in unified CSV but NOT in hierarchy JSON (will be skipped):")
    print(f"  Examples: {sorted(list(annotations_only))[:10]}")

if not_found_sections:
    print(f"\n⚠️  {len(not_found_sections)} sections from unified CSV were not found in hierarchy JSON files")
    print(f"  Examples: {not_found_sections[:5]}")


Loaded annotations for 212 sections from unified CSV

Loading sections for act 2010_15 from hierarchy JSON files...
  Found 4 chunk files

Loading sections for act 1989_41 from hierarchy JSON files...
  Found 9 chunk files

Loading sections for act 2014_6 from hierarchy JSON files...
  Found 5 chunk files

Loaded 212 sections from hierarchy JSON files


In [3]:
# Get experimental sections (first 10 used in experiments)
experimental_sections = set()

with open(unified_csv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= 10:
            break
        experimental_sections.add(row['legislation_id'])

print(f"Experimental sections (will go to test set): {len(experimental_sections)}")
print("\nExperimental sections:")
for s in sorted(experimental_sections):
    print(f"  {s}")


Experimental sections (will go to test set): 10

Experimental sections:
  1989_41_section-1
  1989_41_section-10
  1989_41_section-100
  1989_41_section-101
  1989_41_section-102
  1989_41_section-104
  1989_41_section-104A
  1989_41_section-105
  1989_41_section-106
  1989_41_section-107


## Step 4: Split into Train/Test Sets


In [4]:
# Get sections that have both section JSON and annotations
valid_sections = set(all_sections.keys()) & set(all_annotations.keys())
print(f"Sections with both JSON and annotations: {len(valid_sections)}")

# Remove experimental sections (they go to test)
valid_sections = valid_sections - experimental_sections
print(f"Sections available for train/test split: {len(valid_sections)}")

# Split remaining sections 80/20
valid_list = list(valid_sections)
random.shuffle(valid_list)

train_ratio = 0.8
split_idx = int(len(valid_list) * train_ratio)
train_sections = valid_list[:split_idx]
test_sections = valid_list[split_idx:]

# Add experimental sections to test set
experimental_in_test = [s for s in experimental_sections if s in all_sections and s in all_annotations]
test_sections.extend(experimental_in_test)

print(f"\nTrain set: {len(train_sections)} sections")
print(f"Test set: {len(test_sections)} sections ({len(experimental_in_test)} experimental + {len(test_sections) - len(experimental_in_test)} others)")


Sections with both JSON and annotations: 212
Sections available for train/test split: 202

Train set: 161 sections
Test set: 51 sections (10 experimental + 41 others)


## Step 5: Load Prompt Template


In [5]:
# Load prompt template
prompt_template = prompt_path.read_text(encoding='utf-8')
print("Prompt template loaded:")
print(prompt_template[:200] + "...")


Prompt template loaded:
You are an expert legal annotator. Extract every regulative norm (obligation, prohibition, permission) that is explicitly stated or strongly implied in the legislative material provided.

Return ONLY ...


## Step 6: Create JSONL Files


In [6]:
def create_jsonl_entry(section_json, annotations, prompt_template):
    """Create a single JSONL entry in OpenAI fine-tuning format."""
    # Format section JSON as string
    section_json_str = json.dumps(section_json, ensure_ascii=False, indent=2)
    
    # Format annotations as JSON array string
    annotations_json_str = json.dumps(annotations, ensure_ascii=False, indent=2)
    
    # Build user prompt
    user_prompt = prompt_template.replace("{{DOCUMENT_JSON}}", section_json_str)
    user_prompt = user_prompt.replace("{{SOURCE_PATH}}", "training_data")
    
    # Create messages
    messages = [
        {
            "role": "system",
            "content": "You are an expert legal annotator who extracts and formats obligations, "
                      "prohibitions, and permissions from legislative text. Respond with precise "
                      "and well-structured JSON only."
        },
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": annotations_json_str
        }
    ]
    
    return {"messages": messages}


def create_jsonl_file(section_ids, all_sections, all_annotations, prompt_template, output_path):
    """Create a JSONL file from section IDs."""
    with open(output_path, 'w', encoding='utf-8') as f:
        for section_id in section_ids:
            section_json = all_sections[section_id]
            annotations = all_annotations[section_id]
            
            entry = create_jsonl_entry(section_json, annotations, prompt_template)
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    
    print(f"Created {output_path} with {len(section_ids)} entries")


In [7]:
# Create training JSONL
train_path = output_dir / "train.jsonl"
create_jsonl_file(train_sections, all_sections, all_annotations, prompt_template, train_path)


Created /Users/safiakanwal/Desktop/Work/Swansea/Projects/automatedAnnotationOfNorms/automatedAnnotationOfNorms/automator/training_data/train.jsonl with 161 entries


In [8]:
# Create test JSONL
test_path = output_dir / "test.jsonl"
create_jsonl_file(test_sections, all_sections, all_annotations, prompt_template, test_path)


Created /Users/safiakanwal/Desktop/Work/Swansea/Projects/automatedAnnotationOfNorms/automatedAnnotationOfNorms/automator/training_data/test.jsonl with 51 entries


## Step 7: Preview Training Data


In [9]:
# Preview first training example
with open(train_path, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    first_entry = json.loads(first_line)
    
print("First training example:")
print(json.dumps(first_entry, indent=2, ensure_ascii=False))


First training example:
{
  "messages": [
    {
      "role": "system",
      "content": "You are an expert legal annotator who extracts and formats obligations, prohibitions, and permissions from legislative text. Respond with precise and well-structured JSON only."
    },
    {
      "role": "user",
      "content": "You are an expert legal annotator. Extract every regulative norm (obligation, prohibition, permission) that is explicitly stated or strongly implied in the legislative material provided.\n\nReturn ONLY a JSON array. Each element must look like this:\n{\n  \"main_section\": \"section identifier where the norm is primarily sourced\",\n  \"type\": \"OBLIGATORY\" | \"PROHIBITED\" | \"PERMITTED\",\n  \"for\": \"actor(s) described in the norm\",\n  \"to\": \"action required/forbidden/permitted\",\n  \"conditions\": [\n    {\n      \"type\": \"WHEN/IF/WHERE\" | \"ONLY IF\" | \"BEFORE\" | \"AFTER\" | \"SUBJECT TO\" | \"UNLESS\",\n      \"text\": \"verbatim or closely paraphrased

In [21]:
# Statistics
print("\n" + "="*80)
print("TRAINING DATA SUMMARY")
print("="*80)
print(f"Total sections available: {len(valid_sections) + len(experimental_in_test)}")
print(f"Train set: {len(train_sections)} sections")
print(f"Test set: {len(test_sections)} sections")
print(f"  - Experimental sections in test: {len(experimental_in_test)}")
print(f"  - Other sections in test: {len(test_sections) - len(experimental_in_test)}")
print(f"\nTrain ratio: {len(train_sections) / (len(train_sections) + len(test_sections)) * 100:.1f}%")
print(f"Test ratio: {len(test_sections) / (len(train_sections) + len(test_sections)) * 100:.1f}%")
print("="*80)



TRAINING DATA SUMMARY
Total sections available: 212
Train set: 161 sections
Test set: 51 sections
  - Experimental sections in test: 10
  - Other sections in test: 41

Train ratio: 75.9%
Test ratio: 24.1%


## Step 8: Save Section IDs for Reference


In [22]:
# Save section IDs
train_ids_path = output_dir / "train_section_ids.txt"
test_ids_path = output_dir / "test_section_ids.txt"

with open(train_ids_path, 'w', encoding='utf-8') as f:
    for section_id in sorted(train_sections):
        f.write(f"{section_id}\n")

with open(test_ids_path, 'w', encoding='utf-8') as f:
    for section_id in sorted(test_sections):
        f.write(f"{section_id}\n")

print(f"Saved section IDs to {train_ids_path} and {test_ids_path}")


Saved section IDs to /Users/safiakanwal/Desktop/Work/Swansea/Projects/automatedAnnotationOfNorms/automatedAnnotationOfNorms/automator/training_data/train_section_ids.txt and /Users/safiakanwal/Desktop/Work/Swansea/Projects/automatedAnnotationOfNorms/automatedAnnotationOfNorms/automator/training_data/test_section_ids.txt
